<a href="https://colab.research.google.com/github/vonmatterlorenzohorn/computer_vision_cs4250_8....5/blob/main/exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision
## Exercise Sheet 1: Imaging
### Erhardt Barth / Christoph Linse / Manuel Laufer / Kathleen Anderson
Universität zu Lübeck, Institut für Neuro- und Bioinformatik



## Group Members:

1. Saurabh Kulkarni (816045)
2. Alina Khan (819206)
3. Aayushi Chavan (822139)
4. Mahidhar Kollipara (819800)
5. Donuru Umakanth Reddy (802457)
6. Siddhanth Kate (822990)

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
  if os.getcwd() == '/content':
    !git clone 'https://github.com/inb-luebeck/cs4250.git'
    os.chdir('cs4250')

In [ ]:
import cv2 # open cv
import matplotlib.pyplot as plt # plotting tools
import numpy as np # matrix, array operations

from os.path import join # combine different items to a path
from os import getcwd, listdir # shows the current directory, lists items in a directory

# show plots when running cell
%matplotlib inline

## Exercise 1.1
### Loading and displaying images in Python


In [ ]:
# TODO: define image path
image_path = 'data/exercise_1/clown.png'

# TODO: read image
img_bgr = cv2.imread('data/exercise_1/clown.png')

# TODO: convert image to grayscale
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# TODO: display image
plt.imshow(img_gray, cmap='gray')
plt.axis('off')

## Exercise 1.2
### Image gradients
Problem: By default, plt.imshow independently auto-scales the color map of each individual subplot to fit its own local minimum and maximum intensity values. Because the absolute intensity variations in the horizontal ($d_x$) and vertical ($d_y$) directions can differ significantly, independent scaling visually distorts the relative strength of the edges when comparing the subplots side-by-side. <br>
Solution: Calculate a global minimum (vmin_val) and maximum (vmax_val) across both gradient arrays simultaneously, and explicitly pass these global limits into the vmin and vmax arguments of plt.imshow for all subplots.

In [ ]:
def load_gray_normalized(image_path):
    pass

In [ ]:
# TODO: load image
img_normalized = load_gray_normalized('data/exercise_1/clown.png')
# TODO: define kernels
# detect vertical edges
kernel_dx = np.array([[-1, 0, 1]], dtype=np.float32)
# detect horizontal edges
kernel_dy = np.array([[-1], [0], [1]], dtype=np.float32)
# TODO: filter images
def load_gray_normalized(clown):
    img_bgr = cv2.imread(clown)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return img_gray.astype(np.float32) / 255.0
grad_x = cv2.filter2D(img_normalized, ddepth=-1, kernel=kernel_dx)
grad_y = cv2.filter2D(img_normalized, ddepth=-1, kernel=kernel_dy)
# TODO: display images
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
vmin_val = min(grad_x.min(), grad_y.min())
vmax_val = max(grad_x.max(), grad_y.max())

im1 = axes[0].imshow(grad_x, cmap='gray', vmin=vmin_val, vmax=vmax_val)
axes[0].set_title('Gradient $d_x$')
axes[0].axis('off')

im2 = axes[1].imshow(grad_y, cmap='gray', vmin=vmin_val, vmax=vmax_val)
axes[1].set_title('Gradient $d_y$')
axes[1].axis('off')



## Exercise 1.3
### Point operations
Can you identify the exposure problems in the histograms?
ueb131.png is Underexposed: Its histogram distribution is heavily compressed and concentrated toward the left side (values near $0.0$), meaning the pixel mass resides entirely in dark tones. <br> ueb132.png is Overexposed: Its histogram distribution is heavily shifted toward the right side (values near $1.0$), meaning the pixels occupy mostly the brightest highlights.

Why not [0,255]? <br>
The natural logarithm of zero ($\log(0)$) is mathematically undefined ($\lim_{x \to 0^+} \log(x) = -\infty$). Starting the range at $1$ avoids encountering arithmetic runtime errors or infinite outputs during transformation. <br>

Where do the functions change quickly, where do they change slowly? How can you use this knowledge to improve the images? <br>
Logarithmic / Low Gamma Functions ($\gamma < 1$): Change very quickly at low input intensities and flatten out (slow down) at higher values. This behavior expands the dynamic range of deep shadows, mapping closely-grouped dark pixels to a broader spectrum of brighter outputs. Therefore, use these transformations to correct underexposed (dark) images like ueb131.png.Quadratic / High Gamma Functions ($\gamma > 1$): Change very slowly at low input ranges and climb steeply (speed up) near high input ranges. This behavior stretches out mid-to-bright values while suppressing low ones. Therefore, use these transformations to restore contrast and detail to washed-out, overexposed (bright) images like ueb132.png.

In [ ]:
def display_with_hist(image):
    pass

In [ ]:
def normalized_to_uint8(image):
    pass

In [ ]:
def display_with_hist(image, title="Image and Histogram"):
    """Creates two subplots: showing the image and its normalized intensity histogram."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Show Image
    axes[0].imshow(image, cmap='gray', vmin=0, vmax=1 if image.dtype != np.uint8 else 255)
    axes[0].set_title(f"{title} (Visual)")
    axes[0].axis('off')

    # Show Histogram
    # Flatten array to 1D vector for plotting histograms
    axes[1].hist(image.flatten(), bins=256, density=True, color='purple', alpha=0.7)
    axes[1].set_title(f"{title} Histogram")
    axes[1].set_xlabel("Intensity Value")
    axes[1].set_ylabel("Density")

    plt.tight_layout()
    plt.show()

def normalized_to_uint8(image):
    """Converts a [0, 1] floating point image to [0, 255] uint8."""
    # Clip to guard against floating errors outside [0, 1] range before scaling
    clipped = np.clip(image, 0.0, 1.0)
    return (clipped * 255.0).astype(np.uint8)
# TODO: load images
ueb131 = load_gray_normalized('data/exercise_1/ueb131.png')
ueb132 = load_gray_normalized('data/exercise_1/ueb132.png')
# TODO: display images with histograms
display_with_hist(ueb131)
display_with_hist(ueb132)

In [ ]:
# TODO: convert images to uint8
ueb131_uint8 = normalized_to_uint8(ueb131)
ueb132_uint8 = normalized_to_uint8(ueb132)
# TODO: equalize histogram and display
ueb131_equalized = cv2.equalizeHist(ueb131_uint8)
ueb132_equalized = cv2.equalizeHist(ueb132_uint8)

display_with_hist(ueb131_equalized, title="ueb131 Equalized (uint8)")
display_with_hist(ueb132_equalized, title="ueb132 Equalized (uint8)")

In [ ]:
# TODO: display logarithmic function
x_lin = np.linspace(0.0, 1.0, 500)
x_log_domain = np.linspace(1.0, 255.0, 255)

plt.figure(figsize=(12, 5))
# TODO: display quadratic function
plt.subplot(1, 2, 1)
plt.plot(x_lin, x_lin**2, 'r-', linewidth=2)
plt.title("Power Function: $f(x) = x^2$ in $[0, 1]$")
plt.xlabel("Input Intensity")
plt.ylabel("Output Intensity")
plt.grid(True)

In [ ]:
# TODO: transform and display ueb131.png
gamma_brighten = 0.4
ueb131_corrected = np.power(ueb131, gamma_brighten)

display_with_hist(ueb131_corrected, title="ueb131 Enhanced (Gamma = 0.4)")

In [ ]:
# TODO: transform and display ueb132.png
ueb132_corrected = np.power(ueb132, 2.0)

display_with_hist(ueb132_corrected, title="ueb132 Enhanced (Quadratic $x^2$)")